# Reconstructing Arctic sea-ice velocity with a neural network: an Observing System Design tutorial

*OAISIS — AI-based Observing System Design for polar sea ice*

**The problem.** Some quantities are known everywhere: winds from reanalysis, ice concentration from satellites, the date, the map. Others are only measured where an instrument happens to be — sea-ice velocity from a drifting buoy, say. If we can learn how the first group relates to the second *at the instrument locations*, we can predict the second everywhere.

![](https://raw.githubusercontent.com/lahoffman/ml-tutorial-OAISIS/main/figures/problem.png)

**Training.** A neural network learns the relation inputs → ice velocity *at the observation locations* — here the tracks of real Ice-Tethered Profilers, with climate-model output standing in for the measurements. How well the result works depends on where the instruments were, and that dependence is what **Observing System Design** studies: each observing system gives a different trained network — ① a single ITP (ITP 95, two months north of the Laptev Sea), ② the same ITP plus a second one elsewhere (ITP 5, six months drifting toward Fram Strait).

![](https://raw.githubusercontent.com/lahoffman/ml-tutorial-OAISIS/main/figures/training.png)

**Reconstruction.** Each trained network is then given the inputs on the *full grid* and predicts ice velocity everywhere. Because the "world" is a climate model, the true velocity is known everywhere too, so the prediction can be scored honestly (①, ②) — and the two scores compared (③): did the added observations improve the reconstruction?

![](https://raw.githubusercontent.com/lahoffman/ml-tutorial-OAISIS/main/figures/reconstruction.png)

The inputs are only ever things that would be available on the full grid in reality. The two networks in the figures are real: they are the `ITP 95` and `ITP 95 + 5` observing systems of experiment A in section 4 of this notebook.

**How to use this notebook.** Run it top to bottom once. The code is short on purpose: the functions live in the small `oaisis` package in this repository (`oaisis/data.py`, `ml.py`, `plots.py`, `metrics.py`, `selectors.py`, `experiments.py` — open them if you want to see *how*). You only need to edit **two cells**, both marked **CHANGE ME**: the hyperparameters (section 2.3) and your own experiment (section 5). Everything else adapts.

## 0. Setup

Everything the notebook needs is in this repository: one NetCDF file in `data/`, the `oaisis` package, and the packages in `requirements.txt`.
On **Google Colab** the first cell installs the package and fetches the data file; on **Binder** or your own machine (`pip install -r requirements.txt -e .`) it does nothing.

In [ ]:
# Running on Google Colab? Install the tutorial package and fetch the data file (about a minute).
# Does nothing on Binder or on your own machine, where everything is already in the repository.
import os, sys
if "google.colab" in sys.modules:
    %pip install -q cartopy cmocean git+https://github.com/lahoffman/ml-tutorial-OAISIS.git
    if not os.path.exists("data/oaisis_tutorial_IPSL.nc"):
        !mkdir -p data && wget -q -O data/oaisis_tutorial_IPSL.nc https://raw.githubusercontent.com/lahoffman/ml-tutorial-OAISIS/main/data/oaisis_tutorial_IPSL.nc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from oaisis import data, plots, ml, metrics, selectors, experiments

data.ds          # the dataset: every variable carries long_name, units and a comment

## 1. Synthetic observations

### 1.1 About the data

**Ice-Tethered Profilers (ITPs)** drift with the sea ice and report their position daily, so their tracks give us a realistic *sampling pattern* of the Arctic (Toole et al., 2011; Krishfield et al., 2008; https://www.whoi.edu/itp).

We don't use the ITP measurements themselves. Instead we take daily output from CMIP6 historical simulations and pick, for every ITP position/day, the model value at the nearest grid cell and date. The result is a set of **synthetic observations** at the ITP locations where we know the truth everywhere else as well — exactly what is needed to test an observing system.

The dataset has two parts, both from the CMIP6 historical simulation of **IPSL-CM6A-LR**:

* `obs_*` — 31 ensemble members × 11 242 ITP samples (2004–2014): the synthetic observations. Dimensions `(member, sample)`, plus `obs_lat`, `obs_lon`, `obs_time`, `obs_track` per sample.
* `map_*` — ten days of Arctic fields from member r1 on the model's native ocean grid, used as the truth for the reconstruction. Dimensions `(y, x, time)`.

Variables: `uas`, `vas` 10 m wind [m/s]; `sic` sea-ice concentration [%]; `siu`, `siv` sea-ice velocity [m/s]. All velocity components are **eastward / northward**. (CMIP6 stores ice velocity along the model grid's own axes, which on IPSL's tripolar grid are rotated by tens of degrees from east; the preprocessing rotated them, see `preprocessing/README.md`.)

### 1.2 A look at the synthetic observations

Where the ITPs were (coloured by day of year — the tracks are continuous in time), and what the model says the ice and the wind were doing there, for one ensemble member.

In [ ]:
plots.obs_overview(member="r1i1p1f1")

In [ ]:
plots.relationships(member="r1i1p1f1")

Ice drifts at roughly 2 % of the wind speed (the classic Nansen rule), and the correlation is high — a good sign that wind is a strong predictor. The scatter around the line is what ice concentration, season and location have to explain.

### 1.3 Building the machine-learning dataset

**Inputs (features):** `uas`, `vas`, `sic`, `doy`, `lat`, `cos_lon`, `sin_lon` — things that are known everywhere. (`wind_speed` is also available; you choose the inputs in section 2.3.)
**Targets:** `siu`, `siv`.

Longitude goes in as its cosine and sine rather than as a number: `lon/180` jumps from +1 to −1 at the dateline, and a network fed that will happily produce a jump in ice velocity across 180° W.

**Train / validation / test split.** We split by **ensemble member**, not by random sample: consecutive days along an ITP track are strongly correlated, so a random split would leak information from train to test and make the model look better than it is. Training: members r2–r21; validation: r22–r26; test: r1 and r27–r31. Member r1 provides the reconstruction maps in section 3, so it is kept out of training.

**Feature scaling.** Inputs with very different ranges (wind ±20 m/s, concentration 0–100 %, latitude 70–90°) would otherwise be weighted unevenly. The winds are standardized with the **training** mean and standard deviation (using anything else would be data leakage); inputs with a natural range are simply scaled to it: `sic/100`, `doy/365`, `lat/90`, and `cos_lon`/`sin_lon` are already in [−1, 1]. The targets are standardized too, so the loss is in units of "standard deviations of ice velocity". Note the scaled range of `sic`: ITPs sit in the pack, so the network almost never sees concentrations below 0.8 — keep that in mind at the ice edge later.

`data.prepared()` does the split and the scaling and returns the three tables (`X_tr, Y_tr`, … in physical units; `x_tr, y_tr`, … scaled) and the `scaler` that undoes it. The same function is used on the ITP samples (training) and on the full maps (reconstruction), so the two can never get out of step.

In [ ]:
p = data.prepared()
data.summary(p)

## 2. Neural network

### 2.1 Metrics and loss functions

Velocity is a vector, so we score the vector error $|\mathbf{u}_{\rm pred}-\mathbf{u}_{\rm true}|$ rather than `siu` and `siv` separately — one number, and a model that gets the speed right but the direction wrong is penalised.

* **Skill** $= 1 - \mathrm{RMS}(|\mathbf{u}_{\rm pred}-\mathbf{u}_{\rm true}|)\,/\,\mathrm{RMS}(|\mathbf{u}_{\rm true}-\bar{\mathbf{u}}_{\rm true}|)$: 1 is perfect, 0 is "no better than predicting the mean velocity", negative is worse than that. It penalises bias and wrong amplitude.
* **Correlation** $r$: the vector analogue of Pearson correlation. Pattern agreement only — a prediction can have $r \approx 1$ and still be biased or too weak. Look at both.

These two are how we *judge* a model (`metrics.skill`, `metrics.corr`, `metrics.report`). The **loss function** is different: it is what the network *minimizes* during training, computed on the scaled targets. Keras has the usual ones built in — set any of these as `loss` in the hyperparameters:

| `loss=` | what it is | when |
|---|---|---|
| `"mse"` | mean squared error | the default; punishes large errors hardest |
| `"mae"` | mean absolute error | more tolerant of outliers (fast-ice events) |
| `"huber"` | squared for small errors, absolute for large | a compromise between the two |
| `"log_cosh"` | smooth version of `mae` | similar to huber |
| `"mean_squared_logarithmic_error"` | squared error of log(1 + y) | rarely useful for signed targets like these |
| `metrics.nrmse_loss` | our own: RMSE normalised by the target spread, i.e. 1 − skill per component | same minimum as `mse`, noisier curve |

Any [Keras loss](https://keras.io/api/losses/regression_losses/) works, by its string name or as a function.

### 2.2 A baseline first

Before training anything non-linear: how far does a straight line get? A linear regression on the same inputs is the number to beat. If the network can't improve on it, the network isn't learning anything the physics didn't already tell us.

In [ ]:
linear, baseline = ml.linear_baseline(p)      # test skill of the linear model, and its validation loss for the learning curve

### 2.3 Hyperparameters and inputs

<div style="background:#fff3cd; border:3px solid #f5a742; border-radius:10px; padding:14px 18px; margin:6px 0;">
<span style="font-size:1.6em; font-weight:bold; color:#b35c00;">&#9998; CHANGE ME &nbsp;(1 of 2)</span><br>
<span style="font-size:1.1em;">The <b>inputs</b> the network sees and its <b>hyperparameters</b>. Everything below — training, reconstruction, the experiments in sections 4 and 5 — picks these up automatically. Edit, then run the notebook from here down.</span>
</div>

In [ ]:
# ╔═════════════════════════════════════════════════════════════════════════════╗
# ║  ✎  CHANGE ME (1 of 2) — the inputs and the hyperparameters of the network  ║
# ╚═════════════════════════════════════════════════════════════════════════════╝

# Inputs: any of data.ALL_FEATURES = ["uas", "vas", "sic", "doy", "lat", "cos_lon", "sin_lon", "wind_speed"]
data.use_features(["uas", "vas", "sic", "doy", "lat", "cos_lon", "sin_lon"])

HP = dict(
    n_units=32,        # neurons per hidden layer
    n_layers=2,        # hidden layers
    activation="relu", # "relu", "tanh", "sigmoid", "elu", ...
    dropout=0.0,       # dropout rate (0 = off)
    l2=1e-3,           # L2 weight penalty (0 = off)
    lr=1e-3,           # learning rate
    epochs=30,
    batch_size=256,
    loss="mse",        # "mse", "mae", "huber", "log_cosh", ... or metrics.nrmse_loss   (see 2.1)
)

### 2.4 Train

`ml.train` builds the network from `HP`, fits it on the training members and evaluates the loss on the validation members after every epoch. The loss axis of the learning curve starts at zero on purpose: most of the way from "predict the mean" (loss 1) to the final loss is covered by the straight line (dashed), and the network's contribution is the gap below it. That is what this problem looks like — same-day wind, concentration and position don't determine ice drift beyond that (ocean currents, internal ice stress and inertia are not inputs).

In [ ]:
p = data.prepared()                                   # rebuilt here so that a change of inputs above is picked up
linear, baseline = ml.linear_baseline(p, verbose=False)

model, history = ml.train(p, hp=HP, verbose=1)
model.summary()
plots.history(history, baseline=baseline)

### 2.5 Evaluate on the held-out test members

The model has never seen these ensemble members. We evaluate in *physical* units (m/s) after undoing the scaling.

In [ ]:
Y_pred = ml.predict(model, p.scaler, p.X_te)
Y_lin = p.scaler.y_inv(linear.predict(p.x_te))

metrics.report(p.Y_te, Y_lin,  label="linear")
metrics.report(p.Y_te, Y_pred, label="network")
plots.prediction_scatter(p.Y_te, Y_pred, title="test members:")

## 3. Reconstruction: from ITP tracks to the whole Arctic

The inputs (wind, concentration, date, position) are known at every grid cell of the model. We feed them through the network trained on the ITP samples and compare the result with the model's own ice velocity — which the network has never seen outside the ITP tracks. The `map_*` variables hold IPSL-CM6A-LR member r1 on its native ocean grid (`y` × `x` × `time`), north of 55 °N, NaN over land.

![](https://raw.githubusercontent.com/lahoffman/ml-tutorial-OAISIS/main/figures/sparse_to_global.png)

### 3.1 The inputs everywhere

The target we want everywhere, and the inputs we have everywhere, for one day. (IPSL uses a curvilinear tripolar grid; `plots.polar_pcolormesh` draws it in projected coordinates so that there are no streaks at the dateline and no gap at the grid's north fold.)

In [ ]:
plots.input_maps(t=0)

### 3.2 Reconstruct

Same features → same scaler → same model, on 65 × 360 grid cells × 10 days instead of along the tracks. `ml.reconstruct` returns an array `(y, x, day, [u, v])` in m/s, NaN over land.

In [ ]:
Y_rec = ml.reconstruct(model, p.scaler)
plots.reconstruction_day(Y_rec, t=0)

### 3.3 Skill of the reconstruction

Two views:

* **Pooled** over all ice-covered cells and days: one number, directly comparable to the test score in section 2.
* **Per grid cell** over the ten days: a map of where the reconstruction works. With only a few days per cell the per-cell numbers are noisy and a small bias is enough to push skill below zero, so read the map qualitatively; cells with no ice or no variability are masked.

In [ ]:
metrics.pooled_scores(Y_rec)
s_map, c_map = metrics.cell_scores(Y_rec)
plots.cell_scores(s_map, c_map, title="reconstruction trained on all ITP samples")

The reconstruction scores well below the test score of section 2 — and that is the point of the exercise, not a bug. The ITPs live in thick, slow pack ice (concentration ≈ 95 %, drift σ ≈ 0.09 m/s); the maps also contain the marginal ice zone and early-winter coastal ice (σ ≈ 0.16 m/s; Fram Strait, Barents and Bering Seas, Baffin Bay) that no ITP has ever sampled. The network has to extrapolate there. Where the error is large is where new observations would be most valuable.

## 4. Observing System Design: what if we had different observations?

`experiments.run_experiment` takes a dictionary of *observing systems* — each a selection of ITP samples — retrains the network on each (three seeds, because one training run is noisy), scores the Arctic-wide reconstruction, and draws the results. `experiments.run_sequence` does the same for observing systems that grow one ITP at a time. Two experiments here, and a template for your own in section 5:

* **A — where.** One ITP at a time, in different parts of the Arctic. How much does the location of a single instrument matter?
* **B — how many.** Start from one ITP and add one at a time. How does skill grow, and how many ITPs does it take to approach the skill of using all of them?

### 4.1 Selecting observations

An observing system is a boolean array over the 11 242 ITP samples: `True` for the samples we get to use. The selectors build these arrays from questions you would actually ask; combine them with `&` (and), `|` (or) and `~` (not).

| selector | keeps |
|---|---|
| `selectors.tracks(5, 15)` | the whole tracks of these ITPs (ids on the map below) |
| `selectors.region(lon=(-180, -120), lat=(70, 80))` | samples inside a lon/lat box (dateline-safe) |
| `selectors.near(lat=77, lon=-138, radius_km=75)` | samples within a circle — a stationary "mooring" |
| `selectors.years(2007, 2010)`, `.season("JJA")`, `.months(3, 4)` | samples in those years / months |
| `selectors.every(5)` | every 5th sample (thins the daily record) |
| `selectors.everything()` | all samples — the reference every experiment is compared with |

The ITP tracks, labelled by id (only those with at least 30 days of data):

In [ ]:
plots.tracks(min_days=30)
print("everything():", selectors.describe(selectors.everything()))

### 4.2 Running an experiment

`experiments.run_experiment(name, scenarios)` trains one network per scenario and per seed (with the inputs and hyperparameters of section 2.3, at 15 epochs — the curve is flat by then), always adds the `all` reference, and draws three figures: the sampling maps, the skill bar chart (bar = mean over seeds, whisker = min–max; differences smaller than the whiskers are not results), and maps of where each scenario falls short of the reference. `quick=True` uses one seed and fewer epochs — good for a first look before a full run. Every result is cached: re-running an unchanged selection returns it instantly and says so, which is why the `all` reference is trained only once.

### 4.3 Experiment A — where: one ITP at a time

Three ITPs that drifted through different parts of the Arctic, each used alone — plus the pair from the schematic at the top of the notebook (NN 1 = ITP 95, NN 2 = ITP 95 + ITP 5).

In [ ]:
results_A = experiments.run_experiment("A: where", {
    "ITP 95":     selectors.tracks(95),        # two months north of the Laptev Sea
    "ITP 5":      selectors.tracks(5),         # six months drifting toward Fram Strait
    "ITP 15":     selectors.tracks(15),        # six months north of the Canadian Archipelago
    "ITP 95 + 5": selectors.tracks(95, 5),     # NN 1 -> NN 2 of the schematic
}, hp=HP)

Where the single ITP was matters as much as how long it lasted: ITP 95 has a third of the samples of ITP 5 and reaches about the same skill. Adding ITP 5 to ITP 95 (NN 1 → NN 2 of the schematic) lifts the skill well above either ITP alone: two regions beat one. The difference maps show *where* each network falls short of the full reconstruction.

### 4.4 Experiment B — how many: adding ITPs one at a time

Start from one ITP in the Beaufort Sea and add one track at a time: first the other long Beaufort tracks, then — following a simple design rule — always the long track that is *farthest from everything we already observe* (`experiments.farthest_first_sequence` picks the order). The dashed line is the skill with all ITP samples.

In [ ]:
sequence = experiments.farthest_first_sequence()          # [1, 2, 3, 4, 96, 24, 7, 27, 5, 15]
results_B = experiments.run_sequence("B: how many", sequence, map_steps=(1, 4, 8, 10), hp=HP)

Two things to read off this curve (the axis is zoomed; the changes are larger than the seed spread). First, the three extra Beaufort tracks (steps 2–4, +1 500 samples) buy almost nothing, while the first tracks from *other* regions (steps 5–7, +1 000 samples) lift skill to the all-ITP line: a new region is worth more than more data from a region we already observe. Second, six or seven well-placed ITPs — about a quarter of the samples — do as well as all 105 tracks, and adding more (steps 9–10) changes nothing. That is the answer to "what is the minimum observing system?" for this problem, with the caveat that the dashed line is not an upper bound: the full set is itself heavily concentrated in the Beaufort Gyre. The maps show the reconstruction filling in as ITPs are added.

## 5. Design your own experiment

An experiment is a dictionary: each entry is a **label** (any text — it becomes the bar's name) and a **selection** of ITP samples.

**1. Choose what to select from.** Every selector returns `True`/`False` for each of the 11 242 ITP samples. What you can pick:

| you want … | write | what you can put in |
|---|---|---|
| particular ITPs | `selectors.tracks(5, 15)` | any track ids from the map in section 4.1 (1–105; the 45 labelled ones have ≥ 30 days) |
| a region | `selectors.region(lon=(-180, -120), lat=(75, 85))` | longitudes −180…180 (a box may cross the dateline, e.g. `lon=(150, -150)`), latitudes 71–90; leave out `lon=` or `lat=` for no limit |
| one spot | `selectors.near(lat=77.1, lon=-138, radius_km=75)` | any position; the radius sets how "stationary" the mooring is |
| a period | `selectors.years(2007)`, `selectors.years(2007, 2010)` | 2004–2014 |
| a time of year | `selectors.season("JJA")`, `selectors.months(3, 4)` | `"DJF"`, `"MAM"`, `"JJA"`, `"SON"`, or month numbers 1–12 |
| thinner sampling | `selectors.every(5)` | any integer: keep every n-th sample |
| everything | `selectors.everything()` | the reference — added to every experiment automatically |

**2. Combine them.** `&` means *both*, `|` means *either*, `~` means *not*: `selectors.region(lon=(-180, -120)) & selectors.season("JJA")` is the Beaufort Sea in summer; `~selectors.tracks(1, 2, 3, 4)` is everything except those four ITPs; `selectors.tracks(22) & selectors.years(2010)` is one year of one ITP. Check a selection with `selectors.describe(sel)` before training on it — a selection with fewer than ~50 samples will train but tell you little.

**3. Run it.** `experiments.run_experiment("a name", my_experiment, hp=HP)` trains one network per entry (plus the reference), prints a table and draws three figures: where the samples are, the skill of each system, and maps of where each falls short of the reference. Use `quick=True` first (one seed, ~1 minute); drop it for the full three-seed run (~1 minute per entry) before drawing conclusions — the whiskers tell you which differences are real. For a system that grows step by step, give `experiments.run_sequence("a name", [ids in order of addition], hp=HP)` a list of track ids instead.

The template below asks whether a stationary mooring or a drifting buoy, with a similar number of days, gives the better reconstruction.

<div style="background:#fff3cd; border:3px solid #f5a742; border-radius:10px; padding:14px 18px; margin:6px 0;">
<span style="font-size:1.6em; font-weight:bold; color:#b35c00;">&#9998; CHANGE ME &nbsp;(2 of 2)</span><br>
<span style="font-size:1.1em;">Your <b>observing systems</b>: one line per system, a label and a selection. Add, remove or change lines; keep the dictionary shape. <code>quick=True</code> for a first look, drop it for the real run.</span>
</div>

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ✎  CHANGE ME (2 of 2) — your observing systems (label: selection)       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

my_experiment = {
    "mooring, 9 yr":    selectors.near(lat=77.1, lon=-138, radius_km=75),       # all ITP days within 75 km of one spot (11 ITPs, 594 days)
    "one buoy, 2 yr":   selectors.tracks(22),                                   # the longest single ITP track (746 days)
    "Beaufort, summer": selectors.region(lon=(-180, -120)) & selectors.season("JJA"),
    "north of 85 N":    selectors.region(lat=(85, 90)),
}
for label, sel in my_experiment.items():
    print(f"{label:18s} {selectors.describe(sel)}")

my_results = experiments.run_experiment("my experiment", my_experiment, quick=True, hp=HP)

Some questions to try, each one line in the dictionary:

* **Thinning.** `selectors.every(5)` against `selectors.everything()`: is a daily record needed, or would weekly do? (`every(2)`, `every(10)` …)
* **Seasons.** `season("JJA")` vs `season("DJF")`: the maps are from November–December — does it help to have observed in the same season?
* **Years.** `years(2004, 2008)` vs `years(2010, 2014)`: do early and late ITPs reconstruct the same 2000 field equally well?
* **Regions.** `region(lon=(-180, -120))` (Beaufort only), `~region(lon=(-180, -120))` (everything but the Beaufort), `region(lat=(85, 90))` (the central Arctic only).
* **Mooring vs. buoy.** Change `radius_km`, or pick another spot (`near(lat=83.8, lon=-111)` has 3 tracks over 4.6 years). Then match the number of days: `near(...)` vs `tracks(22) & years(2007, 2008)`.
* **Order matters.** Re-run experiment B with `experiments.run_sequence("B'", [1, 2, 3, 4, 7, 27, 24, 96, 5, 15], hp=HP)` (the same tracks, longest first instead of farthest first). Skill *drops* at step 5, with a large spread between seeds, before recovering at step 6. Why would adding one track near the pole to four Beaufort tracks make things worse? (Hint: look at where track 7 is, and at what the network has to do between the two clusters.)
* **How many seeds are enough?** `experiments.SEEDS = (0, 1, 2, 3, 4)` for two scenarios that looked different. Are they still?
* **Inputs** (back in section 2.3). Drop `sic`, or drop `lat`/`cos_lon`/`sin_lon`, and re-run from there. What does each contribute, and what does that say about what the ITPs are actually "observing"?